# Revenue Forecasting Using Time Series Analysis

Notebook ini dibuat sebagai portofolio Machine Learning untuk GitHub.

## 1. Import Library dan Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/monthly_revenue.csv")
df["month"] = pd.to_datetime(df["month"])
df.head()

## 2. Visualisasi Trend Revenue

In [ ]:
plt.plot(df["month"], df["revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.show()

## 3. Feature Engineering untuk Trend dan Musiman

In [ ]:
df["t"] = np.arange(len(df))
df["month_num"] = df["month"].dt.month
df["sin_12"] = np.sin(2 * np.pi * df["month_num"] / 12)
df["cos_12"] = np.cos(2 * np.pi * df["month_num"] / 12)
df.head()

## 4. Training Model Forecasting

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

train = df.iloc[:-6]
test = df.iloc[-6:]

features = ["t", "sin_12", "cos_12"]
model = LinearRegression()
model.fit(train[features], train["revenue"])

pred = model.predict(test[features])

print("MAE :", round(mean_absolute_error(test["revenue"], pred), 2))
print("MAPE:", round(mean_absolute_percentage_error(test["revenue"], pred), 4))

## 5. Forecast Revenue Periode Berikutnya

In [ ]:
future_dates = pd.date_range(df["month"].max() + pd.offsets.MonthBegin(1), periods=6, freq="MS")
future = pd.DataFrame({"month": future_dates})
future["t"] = np.arange(len(df), len(df) + 6)
future["month_num"] = future["month"].dt.month
future["sin_12"] = np.sin(2 * np.pi * future["month_num"] / 12)
future["cos_12"] = np.cos(2 * np.pi * future["month_num"] / 12)
future["forecast_revenue"] = model.predict(future[features])

display(future[["month", "forecast_revenue"]])

plt.plot(df["month"], df["revenue"], marker="o", label="Actual")
plt.plot(future["month"], future["forecast_revenue"], marker="o", label="Forecast")
plt.title("Revenue Forecast")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.legend()
plt.show()